In [1]:
import sys
sys.path.append(r'X:/')

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import refinitiv.data as rd
import datetime as dt

from Loaders.EikonSpot_class import EikonSpot
from Database.DB_writer import db_writer
from Database.DB_reader import Database

In [3]:
rd.open_session()

<refinitiv.data.session.Definition object at 0x13315e21790 {name='workspace'}>

In [10]:
spot_df = pd.DataFrame()
for country in ['de', 'be', 'nl', 'at', 'fr']:
    start_date = dt.datetime(2019,6,6)
    end_date = dt.datetime(2019,6,10)

    spot_inst = EikonSpot(country, start_date, end_date)

    spot = spot_inst.spot_data()

    df = spot[((spot.index>=dt.datetime(2019,6,7))&
              (spot.index<dt.datetime(2019,6,9)))].copy()
    df = df[['price']]
    df.columns = [country]
    spot_df = pd.concat([spot_df, df], axis=1)

In [5]:
df['check'] = (df['datetime'] - df['datetime'].shift(1)).dt.seconds/3600

In [6]:
df.loc[df['check']!=1.]

,datetime,b_volume,s_volume,volume,price,check
0,2024-06-05,NaN,NaN,NaN,15.01,NaN


In [7]:
start_date = dt.datetime(2024,6,3)
end_date = dt.datetime(2024,6,7)
spot_df = pd.DataFrame()
check_df = pd.DataFrame()
db_r = Database()
for country in ['de', 'dkw', 'dke',
                 'fr','hu','nl','sk', 'si', 'ro']:
    
    aux_inst = EikonSpot(country, start_date, end_date)
    aux = aux_inst.spot_data()
    df = aux[((aux.index>=dt.datetime(2024,6,5))&
              (aux.index<dt.datetime(2024,6,6)))].copy()
    df = df.reset_index()
    df['check'] = (df['datetime'] - df['datetime'].shift(1)).dt.seconds/3600
    aux_check = df.loc[df['check']!=1.].copy()
    if len(aux_check)>1:
        print('FAILED_CHECK: ', country)
        break
        
    else:
        df.to_sql(name="stage_" + country,
                              schema='spot',
                              con=db_r.connection_string,
                              if_exists='replace', index=False)
        db_r.merge_from_staging_to_prod_enum(schema='spot', table=country)
    

Connected to the database postgre

                MERGE INTO "spot"."de" AS tgt
                USING "spot"."stage_de" AS src
                ON (tgt.datetime = src.datetime)
                WHEN MATCHED THEN UPDATE SET
                "datetime" = CASE WHEN src."datetime" IS NOT NULL THEN src."datetime" ELSE tgt."datetime" END,
"b_volume" = CASE WHEN src."b_volume" IS NOT NULL THEN src."b_volume" ELSE tgt."b_volume" END,
"s_volume" = CASE WHEN src."s_volume" IS NOT NULL THEN src."s_volume" ELSE tgt."s_volume" END,
"volume" = CASE WHEN src."volume" IS NOT NULL THEN src."volume" ELSE tgt."volume" END,
"price" = CASE WHEN src."price" IS NOT NULL THEN src."price" ELSE tgt."price" END
                WHEN NOT MATCHED THEN INSERT ("datetime", "b_volume", "s_volume", "volume", "price")
                VALUES (src."datetime", src."b_volume", src."s_volume", src."volume", src."price");
            
Disconnected from the database postgre
Connected to the database postgre

                MERG

In [8]:
aux

,b_volume,s_volume,volume,price
datetime,,,,
2024-06-02 13:00:00,NaN,NaN,NaN,-4.01
2024-06-03 00:00:00,NaN,NaN,NaN,93.60
2024-06-03 01:00:00,NaN,NaN,NaN,82.55
2024-06-03 02:00:00,NaN,NaN,NaN,80.20
2024-06-03 03:00:00,NaN,NaN,NaN,58.08
...,...,...,...,...
2024-06-08 19:00:00,NaN,NaN,NaN,34.23
2024-06-08 20:00:00,NaN,NaN,NaN,32.57
2024-06-08 21:00:00,NaN,NaN,NaN,31.73
